<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_06_model_training/seq2one/stage_06_04_mlp_seq2one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_04 -  SEQ2ONE - Modelo MLP**

El **MLP (Multi-Layer Perceptron)** es una red neuronal **feedforward** que introduce no linealidad en la relación entre la ventana histórica aplanada
(L x N → cantidad de features) y el target escalar.

A diferencia de los modelos lineales (Ridge/Lasso), el MLP puede capturar
interacciones no lineales entre las features, manteniendo una arquitectura
simple y controlable.

En este pipeline, el MLP se utiliza como **primer modelo no lineal**
de referencia, entrenado bajo el enfoque **seq2one**, y evaluado con las mismas
métricas que los modelos anteriores.


# **BLOQUE DE EJECUCIÓN COMPLETO**

## **1. Imports + paths**

In [41]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [42]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **3. Rutas de ventanas seq2one y scalers**

In [43]:
from pathlib import Path
import os

WINDOWS_SEQ2ONE_DIR = Path(
    os.environ.get("WINDOWS_SEQ2ONE_DIR", "data/windows/seq2one/")
)

SCALERS_DIR = Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

window_sizes = [30, 60, 90, 120, 180]
targets = ['delta_60', 'delta_90', 'ret_60', 'ret_90']
splits = ['train', 'valid', 'test']

In [44]:
windows_paths = {}

for w in window_sizes:
    windows_paths[w] = {}

    for t in targets:
        windows_paths[w][t] = {}

        for s in splits:
            path = (
                DRIVE_DIR
                / WINDOWS_SEQ2ONE_DIR
                / f"L{w}"
                / f"windows_{t}_{s}.npz"
            )

            windows_paths[w][t][s] = path

#display(windows_paths)

#Como llamarlo:
#path_train_L60_delta = windows_paths[60]['delta_90']['train']
#print(path_train_L60_delta)

In [45]:
scalers_paths = {}
for t in targets:
  scalers_paths[t] = {}
  path = (
                DRIVE_DIR
                / SCALERS_DIR
                / f"scaler_{t}.pkl"
            )

  scalers_paths[t] = path

display(scalers_paths)

{'delta_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl'),
 'delta_90': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_90.pkl'),
 'ret_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_ret_60.pkl'),
 'ret_90': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_ret_90.pkl')}

## **4. Reproducibilidad**

In [46]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [47]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [48]:
print(compute_seq2one_metrics.__doc__)


    Calcula métricas simples y comparables para modelos seq2one.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    y_pred : np.ndarray
        Valores predichos con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    compute_r2 : bool
        Si True, calcula R² sobre el vector completo.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 en y_true o y_pred al calcular DA.
    allow_seq_inputs_take_last : bool
        Si True, permite inputs 2D (n_samples, seq_len) y toma el último paso [:, -1].
        Útil si algún modelo devuelve secuencia pero usted lo evalúa como many-to-one.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales.
    


## **6. Carga de data windows**

In [49]:
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.
    """

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Usar contexto para cerrar correctamente el archivo
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # Opcional pero recomendable: copiar a memoria
        X = X.copy()
        y = y.copy()

    return X, y


In [50]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [51]:
from typing import Any, Dict, Mapping
from pathlib import Path

# --------------------------------------------------
# Carga completa: ventanas + scaler por window_size y target
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scalers_path: Mapping[str, Path],
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler correspondiente
    a un (window_size, target).

    windows_paths[L][target][split] -> Path
    scalers_path[target] -> Path
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    if target not in scalers_path:
        raise KeyError(f"target='{target}' no existe en scalers_path")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]
    scaler_path = scalers_paths[target]

    # --------------------------
    # 3) Carga
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test,  y_test  = load_npz_windows(test_path)

    scaler = load_scaler(scaler_path)

    # --------------------------
    # 4) Inferir horizonte
    # --------------------------
    horizon = int(target.split("_")[-1])

    # --------------------------
    # 5) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }

In [52]:
import numpy as np

def maybe_flatten_X(X: np.ndarray, *, flatten: bool) -> np.ndarray:
    """
    Si flatten=True y X es 3D (N,L,F) -> (N, L*F)
    Si flatten=False -> retorna X tal cual.
    """
    if not flatten:
        return X
    if X.ndim == 3:
        N, L, F = X.shape
        return X.reshape(N, L * F)
    if X.ndim == 2:
        return X
    raise ValueError(f"X debe ser 2D o 3D, recibí shape={X.shape}")

In [53]:
def create_bundles(window_size, targets: list, windows_paths=windows_paths, scalers_paths=scalers_paths, *, flatten_X=False):

    bundles = []
    for t in targets:
        b = load_windows_and_scaler(
            window_size=window_size,
            target=t,
            windows_paths=windows_paths,
            scalers_path=scalers_paths,
        )
        if flatten_X:
            b["train"]["X"] = maybe_flatten_X(b["train"]["X"], flatten=True)
            b["valid"]["X"] = maybe_flatten_X(b["valid"]["X"], flatten=True)
            b["test"]["X"]  = maybe_flatten_X(b["test"]["X"],  flatten=True)
        bundles.append(b)

    # prints (opcional)
    for b in bundles:
        print(f"H{b['horizon']} Train:", b["train"]["X"].shape, b["train"]["y"].shape)
        print(f"H{b['horizon']} Valid:", b["valid"]["X"].shape, b["valid"]["y"].shape)
        print(f"H{b['horizon']} Test :", b["test"]["X"].shape,  b["test"]["y"].shape)
        print(f"Scaler H{b['horizon']}:", type(b["scaler"]).__name__)

    return tuple(bundles)

In [54]:
#bundle_delta_60, bundle_delta_90 = create_bundles(window_size = 30, targets = ['delta_60', 'delta_90'], windows_paths = windows_paths, scalers_paths = scalers_paths, flatten_X = False)
#bundle_ret_60, bundle_ret_90 = create_bundles(window_size = 30, targets = ['ret_60', 'ret_90'], windows_paths = windows_paths, scalers_paths = scalers_paths)
'''
def run_mlp(window_size: int, *, alpha: float = 1.0, verbose: bool = True):

    size = window_size

    if verbose:
        print("\n" + "=" * 80)
        print(f"RIDGE | SEQ2ONE | WINDOW_SIZE=L{size} | alpha={alpha}")
        print("=" * 80)

    targets = ["delta_60", "delta_90", "ret_60", "ret_90"]
    rows = []

    for target in targets:
        if verbose:
            print(f"\n[BUILD] L{size} | target = '{target}'")


        # crear SOLO 1 bundle (y aplanar X para Ridge)
        (bundle,) = create_bundles(
            window_size=size,
            targets=[target],            # <- SOLO UNO
            windows_paths=windows_paths,
            scalers_paths=scalers_paths,
            flatten_X=True,              # <- MLP necesita 2D
        )
'''


'\ndef run_mlp(window_size: int, *, alpha: float = 1.0, verbose: bool = True):\n\n    size = window_size\n\n    if verbose:\n        print("\n" + "=" * 80)\n        print(f"RIDGE | SEQ2ONE | WINDOW_SIZE=L{size} | alpha={alpha}")\n        print("=" * 80)\n\n    targets = ["delta_60", "delta_90", "ret_60", "ret_90"]\n    rows = []\n\n    for target in targets:\n        if verbose:\n            print(f"\n[BUILD] L{size} | target = \'{target}\'")\n\n\n        # crear SOLO 1 bundle (y aplanar X para Ridge)\n        (bundle,) = create_bundles(\n            window_size=size,\n            targets=[target],            # <- SOLO UNO\n            windows_paths=windows_paths,\n            scalers_paths=scalers_paths,\n            flatten_X=True,              # <- MLP necesita 2D\n        )\n'

NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **7. Sanity Check**

In [55]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [56]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [57]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [58]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

# **DEFINICIÓN DE MODELO**

## **8. Definición del modelo — placeholder**

### **8.1. Modelo MLP (Feedforward) — seq2one**

**Idea básica**

El **MLP (Multi-Layer Perceptron)** es una red neuronal **feedforward**
que aprende una relación **no lineal** entre la entrada aplanada
y el target escalar futuro.

A diferencia de los modelos lineales, el MLP puede capturar
**interacciones no lineales** entre las features, manteniendo una
arquitectura simple y controlable.

Formalmente, el modelo puede representarse como una composición de capas:

$$
\hat{y}_t = f_L\big( W_L \, f_{L-1}( \dots f_1(W_1 x_t + b_1) \dots ) + b_L \big)
$$

donde:
- $x_t \in \mathbb{R}^{1200}$ es la ventana histórica aplanada,
- $f_i(\cdot)$ son funciones de activación no lineales,
- $W_i, b_i$ son pesos y sesgos aprendidos.

---

**Regularización (MLP)**

**Riesgo:** Medio-alto, debido a la mayor capacidad del modelo.

La regularización **no es intrínseca** al MLP y debe definirse explícitamente:

- **Penalización L2 (weight decay):**
  - Controla la magnitud de los pesos.
- **Early stopping:**
  - Detiene el entrenamiento cuando el error en VALID deja de mejorar.
- **Arquitectura controlada:**
  - Pocas capas (1-2).
  - Número reducido de neuronas por capa.
- **Dropout (opcional):**
  - Solo si se observa sobreajuste claro.

La regularización es parte esencial del diseño del modelo y determina
su capacidad de generalización.

---

**Por qué el MLP es relevante en este proyecto**

- Entrada de **alta dimensión**: L x N = **n features**.
- Capacidad para modelar:
  - relaciones no lineales,
  - interacciones entre indicadores técnicos.
- Modelo:
  - más flexible que Ridge/Lasso,
  - más simple y estable que modelos temporales complejos.

El MLP actúa como el **primer baseline no lineal**, sirviendo de puente
entre modelos lineales y arquitecturas temporales más sofisticadas
(LSTM, TCN, Transformer).

---

**Hiperparámetros iniciales**

Para este stage (sin tuning):

- Número de capas ocultas: 1–2
- Neuronas por capa: moderado (por ejemplo, 64–128)
- Función de activación: ReLU
- Regularización L2 (weight decay): fija
- Early stopping: activado
- Dropout: desactivado inicialmente
- **Sin validación interna automática** (la evaluación se realiza externamente en VALID)

El ajuste fino de la arquitectura y la regularización se aborda en etapas posteriores.


### **8.2. Imports (PyTorch) + semillas**

In [59]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [60]:
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

### **8.3. Dataset/DataLoader desde bundle**

In [61]:
def make_loaders_from_bundle(
    bundle,
    *,
    batch_size: int = 4096,
    num_workers: int = 0,
) -> dict:
    """
    Convierte bundle {train/valid/test} a DataLoaders PyTorch.
    Espera X: (n, 1200) y y: (n,)
    """
    loaders = {}

    for split in ["train", "valid", "test"]:
        X = np.asarray(bundle[split]["X"], dtype=np.float32)
        y = np.asarray(bundle[split]["y"], dtype=np.float32).reshape(-1, 1)  # (n,1)

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        shuffle = (split == "train")

        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
        )

    return loaders

In [62]:
#loaders_60 = make_loaders_from_bundle(bundle_60, batch_size=16384)
#loaders_90 = make_loaders_from_bundle(bundle_90, batch_size=16384)

### **8.4. Definición del modelo MLP (simple y controlado)**

In [63]:
class MLPSeq2One(nn.Module):
    def __init__(self, in_dim: int = 1200, hidden_dim: int = 128, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)

### **8.5. Entrenamiento con early stopping (VALID)**

In [64]:
@torch.no_grad()
def evaluate_mse(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    mse_sum = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        pred = model(xb)
        mse_sum += torch.sum((pred - yb) ** 2).item()
        n += yb.numel()
    return mse_sum / max(n, 1)

import time
import torch
import torch.nn as nn

def _ts():
    return time.strftime("%H:%M:%S")

def train_mlp(
    loaders: dict,
    *,
    in_dim: int = 1200,
    hidden_dim: int = 128,
    dropout: float = 0.0,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    device: torch.device,
    verbose: bool = True,
    log_every: int = 0,  # 0 => no log por batch; si pones 50, log cada 50 batches
):
    """
    Entrena un MLP seq2one usando TRAIN y early stopping en VALID (por MSE).
    Logs: inicio/fin, train_loss, valid_mse, mejoras, early-stopping.
    """
    if verbose:
        print(f"[{_ts()}] [TRAIN] START | in_dim={in_dim} hidden_dim={hidden_dim} dropout={dropout} "
              f"lr={lr} wd={weight_decay} max_epochs={max_epochs} patience={patience} "
              f"device={device.type}")

    t_global = time.perf_counter()

    model = MLPSeq2One(in_dim=in_dim, hidden_dim=hidden_dim, dropout=dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_state = None
    best_valid = float("inf")
    bad_epochs = 0

    for epoch in range(1, max_epochs + 1):
        t_epoch = time.perf_counter()

        # -------------------------
        # TRAIN EPOCH
        # -------------------------
        model.train()
        train_loss_sum = 0.0
        train_n = 0

        for b, (xb, yb) in enumerate(loaders["train"], start=1):
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

            # acumular loss para log por época
            bs = yb.numel()
            train_loss_sum += loss.item() * bs
            train_n += bs

            if verbose and log_every and (b % log_every == 0):
                train_loss_avg_so_far = train_loss_sum / max(train_n, 1)
                print(f"[{_ts()}]   epoch={epoch:02d} batch={b:04d} | train_loss_avg={train_loss_avg_so_far:.6f}")

        train_loss_avg = train_loss_sum / max(train_n, 1)

        # -------------------------
        # VALIDATION
        # -------------------------
        #if verbose:
        #    print(f"[{_ts()}]   [VALID] Calculando valid_mse ...")
        t0 = time.perf_counter()

        valid_mse = evaluate_mse(model, loaders["valid"], device)

        dt_valid = time.perf_counter() - t0
        dt_epoch = time.perf_counter() - t_epoch

        # -------------------------
        # EARLY STOPPING LOGIC
        # -------------------------
        improved = valid_mse < (best_valid - 1e-9)
        if improved:
            best_valid = valid_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if verbose:
            flag = "BEST" if improved else f"no_improve({bad_epochs}/{patience})"
            print(
                f"[{_ts()}] epoch={epoch:02d} | train_loss={train_loss_avg:.6f} | "
                f"valid_mse={valid_mse:.6f} | {flag} | dt_valid={dt_valid:.2f}s | dt_epoch={dt_epoch:.2f}s"
            )

        if bad_epochs >= patience:
            if verbose:
                print(f"[{_ts()}] [TRAIN] EARLY STOPPING | patience={patience} | best_valid_mse={best_valid:.6f}")
            break

    if best_state is not None:
        if verbose:
            print(f"[{_ts()}] [TRAIN] Cargando best_state (best_valid_mse={best_valid:.6f}) ...")
        model.load_state_dict(best_state)

    if verbose:
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [TRAIN] END | best_valid_mse={best_valid:.6f} | dt_total={dt_all:.2f}s")

    return model


### **8.6. Predicciones MLP**


In [65]:
import numpy as np
import torch

@torch.no_grad()
def predict_mlp(model, X: np.ndarray, *, device: torch.device, batch_size: int = 32768) -> np.ndarray:
    """
    Predice con un modelo MLP PyTorch en batches.
    Retorna shape (n_samples,)
    """
    model.eval()

    X = np.asarray(X, dtype=np.float32)
    n = X.shape[0]
    preds = []

    for i in range(0, n, batch_size):
        xb = torch.from_numpy(X[i:i+batch_size]).to(device)
        yb = model(xb).squeeze(-1)          # (batch,)
        preds.append(yb.detach().cpu().numpy())

    return np.concatenate(preds, axis=0)


## **9. Métricas ML**

In [66]:
import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    horizon: int,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])

In [67]:
def get_metrics_torch(bundle, model, *, device) -> tuple[dict, dict]:
  # -------- VALID --------
  X_valid = bundle["valid"]["X"]
  y_valid = bundle["valid"]["y"]
  y_pred_valid = predict_mlp(model, X_valid, device=device)
  metrics_valid = compute_seq2one_metrics(y_valid, y_pred_valid, compute_r2=True)

  # -------- TEST --------
  X_test = bundle["test"]["X"]
  y_test = bundle["test"]["y"]
  y_pred_test = predict_mlp(model, X_test, device=device)
  metrics_test  = compute_seq2one_metrics(y_test, y_pred_test,  compute_r2=True)

  return metrics_valid, metrics_test

## **10. Gestión de dataset de métricas**

In [68]:
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [69]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"seq2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

## **11. Ejecución completa**

In [70]:
import pandas as pd
import gc
import time

def _ts():
    return time.strftime("%H:%M:%S")

def run_mlp(window_size: int, *, weight_decay: float = 1e-4, verbose: bool = True):

    size = window_size
    n_features = 36

    if verbose:
        print("\n" + "=" * 80)
        print(f"[{_ts()}] MLP | SEQ2ONE | WINDOW_SIZE=L{size} | in_dim={size*n_features} | weight_decay={weight_decay}")
        print("=" * 80)

    targets = ["delta_60", "delta_90", "ret_60", "ret_90"]
    rows = []

    t_global = time.perf_counter()

    for i, target in enumerate(targets, start=1):
        t_target = time.perf_counter()

        if verbose:
            print(f"\n[{_ts()}] [{i}/{len(targets)}] START target='{target}' | L{size}")

        # -------------------------
        # BUILD BUNDLE
        # -------------------------
        if verbose:
            print(f"[{_ts()}]   [BUILD] Creando bundle (flatten_X=True) ...")
        t0 = time.perf_counter()

        (bundle,) = create_bundles(
            window_size=size,
            targets=[target],
            windows_paths=windows_paths,
            scalers_paths=scalers_paths,
            flatten_X=True,  # MLP necesita 2D
        )

        if verbose:
            dt = time.perf_counter() - t0
            # (opcional) si existe el shape, lo mostramos
            try:
                xshape = bundle["train"]["X"].shape
                yshape = bundle["train"]["y"].shape
                print(f"[{_ts()}]   [BUILD] OK | train X={xshape} y={yshape} | dt={dt:.2f}s")
            except Exception:
                print(f"[{_ts()}]   [BUILD] OK | dt={dt:.2f}s")

        # -------------------------
        # LOADERS
        # -------------------------
        if verbose:
            print(f"[{_ts()}]   [LOADERS] Creando DataLoaders ...")
        t0 = time.perf_counter()

        loaders = make_loaders_from_bundle(
            bundle,
            batch_size=16384
        )

        if verbose:
            dt = time.perf_counter() - t0
            # si los loaders exponen dataset length, lo imprimimos
            try:
                ntr = len(loaders["train"].dataset)
                nva = len(loaders["valid"].dataset)
                nte = len(loaders["test"].dataset)
                print(f"[{_ts()}]   [LOADERS] OK | n(train/valid/test)=({ntr}/{nva}/{nte}) | dt={dt:.2f}s")
            except Exception:
                print(f"[{_ts()}]   [LOADERS] OK | dt={dt:.2f}s")

        # -------------------------
        # TRAIN
        # -------------------------
        if verbose:
            print(f"[{_ts()}]   [TRAIN] Iniciando entrenamiento ...")
        t0 = time.perf_counter()

        model = train_mlp(
            loaders,
            in_dim=size * n_features,
            hidden_dim=128,
            dropout=0.0,
            lr=1e-3,
            weight_decay=weight_decay,
            max_epochs=30,
            patience=5,
            device=device
        )

        if verbose:
            dt = time.perf_counter() - t0
            print(f"[{_ts()}]   [TRAIN] FIN entrenamiento | dt={dt:.2f}s")

        # liberar TRAIN (opcional)
        if verbose:
            print(f"[{_ts()}]   [MEM] Liberando bundle['train'] y gc.collect() ...")
        del bundle["train"]
        gc.collect()

        # -------------------------
        # PRED + METRICS
        # -------------------------
        if verbose:
            print(f"[{_ts()}]   [PRED] Iniciando predicciones + cálculo de métricas (valid/test) ...")
        t0 = time.perf_counter()

        metrics_valid, metrics_test = get_metrics_torch(bundle, model, device=device)

        if verbose:
            dt = time.perf_counter() - t0
            print(f"[{_ts()}]   [METRICS] OK (valid/test) | dt={dt:.2f}s")

        # -------------------------
        # DF APPEND
        # -------------------------
        if verbose:
            print(f"[{_ts()}]   [DF] Agregando filas a la tabla de métricas ...")
        t0 = time.perf_counter()

        rows.append(metrics_to_df(
            metrics_valid,
            model="mlp",
            split="valid",
            horizon=bundle["horizon"],
            window_size=bundle["window_size"],
            target=bundle["target"],
        ))

        rows.append(metrics_to_df(
            metrics_test,
            model="mlp",
            split="test",
            horizon=bundle["horizon"],
            window_size=bundle["window_size"],
            target=bundle["target"],
        ))

        if verbose:
            dt = time.perf_counter() - t0
            print(f"[{_ts()}]   [DF] OK | dt={dt:.2f}s")

        # -------------------------
        # CLEANUP
        # -------------------------
        if verbose:
            print(f"[{_ts()}]   [CLEAN] Liberando objetos (bundle/model/metrics/loaders) ...")
        del bundle, model, metrics_valid, metrics_test, loaders
        gc.collect()

        if verbose:
            dt_target = time.perf_counter() - t_target
            print(f"[{_ts()}] [{i}/{len(targets)}] DONE target='{target}' | dt_total={dt_target:.2f}s")

    # -------------------------
    # FINAL DF
    # -------------------------
    if verbose:
        print(f"\n[{_ts()}] [FINAL] Concatenando resultados ...")
    t0 = time.perf_counter()

    df_mlp_metrics = (
        pd.concat(rows, ignore_index=True)
          .sort_values(["window_size", "target", "split", "horizon_min", "model"])
          .reset_index(drop=True)
    )

    if verbose:
        dt = time.perf_counter() - t0
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [FINAL] OK | rows={len(df_mlp_metrics)} | dt_concat={dt:.2f}s | dt_total={dt_all:.2f}s")
        print(df_mlp_metrics[["window_size", "target", "split", "horizon_min", "model"]]
              .drop_duplicates()
              .to_string(index=False))

    return df_mlp_metrics


In [71]:
def run_mlp_incremental(
    window_sizes: list[int],
    *,
    dropout: float = 0.0,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    name: str = "mlp",  # -> seq2one_ridge_metrics.parquet
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose: bool = True,
) -> pd.DataFrame:

    # 1) Cargar si existe
    df_all = load_seq2one_metrics_if_exists(name=name, base_dir=base_dir)

    # 2) Asegurar columna w_decay
    if df_all.empty:
        df_all = pd.DataFrame(columns=[
            "model","split","window_size","target","horizon_min","MAE","RMSE","R2","DA","w_decay"
        ])
    if "w_decay" not in df_all.columns:
        df_all["w_decay"] = pd.NA

    key_cols = ["model","w_decay","window_size","target","split","horizon_min"]

    # 3) Normalizar tipos (evita falsos mismatches)
    if len(df_all):
        df_all["window_size"] = pd.to_numeric(df_all["window_size"], errors="coerce").astype("Int64")
        df_all["horizon_min"] = pd.to_numeric(df_all["horizon_min"], errors="coerce").astype("Int64")

    # 4) Loop por window_size
    for ws in window_sizes:

        # Si ya existen todas las filas esperadas para este ws, saltar
        # Esperadas: 4 targets x 2 splits (valid/test) = 8 filas para mlp (siempre)
        # (Esto asume que tu run_mlp devuelve valid+test para los 4 targets)
        df_ws = df_all[(df_all["model"] == "mlp") & (df_all["w_decay"] == weight_decay) & (df_all["window_size"] == ws)]
        if len(df_ws) >= 8:
            if verbose:
                print(f"[SKIP] L{ws}: ya hay {len(df_ws)} filas (mlp w_decay={weight_decay}).")
            continue

        if verbose:
            print("\n" + "="*90)
            print(f"[RUN] MLP incremental | L{ws} |in_dim={ws*36}| dropout={dropout} | lr={lr} | w_decay={weight_decay}")  ###################
            print("="*90)

        # 5) Entrenar y obtener métricas para ESTE ws
        df_new = run_mlp(ws, weight_decay=weight_decay, verbose=verbose).copy() #####################
        df_new["w_decay"] = weight_decay  # agregar

        # 6) Filtrar filas ya existentes (anti-duplicados)
        # Creamos un "key" para comparar rápido
        existing_keys = set(tuple(x) for x in df_all[key_cols].dropna().values)
        mask_keep = [tuple(row) not in existing_keys for row in df_new[key_cols].values]
        df_new = df_new.loc[mask_keep].copy()

        if df_new.empty:
            if verbose:
                print(f"[INFO] L{ws}: no había filas nuevas para agregar.")
            continue

        # 7) Merge + dedupe por seguridad
        df_all = pd.concat([df_all, df_new], ignore_index=True)
        df_all = df_all.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)

        # 8) Guardar checkpoint con TU función
        save_seq2one_metrics(df_all, name=name, base_dir=base_dir)

        if verbose:
            print(f"[OK] Checkpoint guardado. Total rows={len(df_all)}")

    return df_all

In [72]:
#df_mlp_all_sizes = load_seq2one_metrics_if_exists(name="mlp", base_dir="/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics")

In [73]:
#MLP_ALL_TRAIN = '''
df_mlp_all_sizes = run_mlp_incremental(
    window_sizes=[30, 60, 90, 120, 180],
    name="mlp",   # genera seq2one_ridge_metrics.parquet
    verbose=True,
)
#'''


[RUN] MLP incremental | L30 |in_dim=1080| dropout=0.0 | lr=0.001 | w_decay=0.0001

[16:03:31] MLP | SEQ2ONE | WINDOW_SIZE=L30 | in_dim=1080 | weight_decay=0.0001

[16:03:31] [1/4] START target='delta_60' | L30
[16:03:31]   [BUILD] Creando bundle (flatten_X=True) ...
H60 Train: (357504, 1080) (357504,)
H60 Valid: (76440, 1080) (76440,)
H60 Test : (76832, 1080) (76832,)
Scaler H60: StandardScaler
[16:03:36]   [BUILD] OK | train X=(357504, 1080) y=(357504,) | dt=5.21s
[16:03:36]   [LOADERS] Creando DataLoaders ...
[16:03:36]   [LOADERS] OK | n(train/valid/test)=(357504/76440/76832) | dt=0.00s
[16:03:36]   [TRAIN] Iniciando entrenamiento ...
[16:03:36] [TRAIN] START | in_dim=1080 hidden_dim=128 dropout=0.0 lr=0.001 wd=0.0001 max_epochs=30 patience=5 device=cuda
[16:03:42] epoch=01 | train_loss=3166.153858 | valid_mse=2694.008320 | BEST | dt_valid=1.01s | dt_epoch=5.73s
[16:03:48] epoch=02 | train_loss=2942.028783 | valid_mse=2596.009681 | BEST | dt_valid=1.01s | dt_epoch=5.93s
[16:03:54] 

/tmp/ipython-input-3761228368.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat([df_all, df_new], ignore_index=True)


[OK] Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics/seq2one_mlp_metrics.parquet
[OK] Checkpoint guardado. Total rows=8

[RUN] MLP incremental | L60 |in_dim=2160| dropout=0.0 | lr=0.001 | w_decay=0.0001

[16:16:15] MLP | SEQ2ONE | WINDOW_SIZE=L60 | in_dim=2160 | weight_decay=0.0001

[16:16:15] [1/4] START target='delta_60' | L60
[16:16:15]   [BUILD] Creando bundle (flatten_X=True) ...
H60 Train: (330144, 2160) (330144,)
H60 Valid: (70590, 2160) (70590,)
H60 Test : (70952, 2160) (70952,)
Scaler H60: StandardScaler
[16:16:33]   [BUILD] OK | train X=(330144, 2160) y=(330144,) | dt=17.99s
[16:16:33]   [LOADERS] Creando DataLoaders ...
[16:16:33]   [LOADERS] OK | n(train/valid/test)=(330144/70590/70952) | dt=0.00s
[16:16:33]   [TRAIN] Iniciando entrenamiento ...
[16:16:33] [TRAIN] START | in_dim=2160 hidden_dim=128 dropout=0.0 lr=0.001 wd=0.0001 max_epochs=30 patience=5 device=cuda
[16:16:40] epoch=01 | train_loss=3004.260432 | valid_mse=2320.031676 | BES

## **11. Observaciones**

In [74]:
df_mlp_all_sizes

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,w_decay
0,mlp,test,30,delta_60,60,50.232999,77.094085,0.264331,0.651685,0.0001
1,mlp,valid,30,delta_60,60,32.880210,46.714091,0.273345,0.661028,0.0001
2,mlp,test,30,delta_90,90,64.345231,97.704426,0.207761,0.632984,0.0001
3,mlp,valid,30,delta_90,90,41.200099,58.197989,0.250902,0.649046,0.0001
4,mlp,test,30,ret_60,60,0.004352,0.022573,-22.849062,0.572902,0.0001
5,mlp,valid,30,ret_60,60,0.002274,0.003187,-0.100169,0.592594,0.0001
6,mlp,test,30,ret_90,90,0.009000,0.020630,-12.426419,0.535364,0.0001
7,mlp,valid,30,ret_90,90,0.004096,0.005877,-1.490132,0.562500,0.0001
8,mlp,test,60,delta_60,60,38.923520,63.419347,0.496504,0.738393,0.0001
9,mlp,valid,60,delta_60,60,24.081825,37.120464,0.550355,0.758303,0.0001


**1. Comportamiento del MLP con target `delta`**

1.1 Mejora al aumentar la ventana

Para `delta_60` (TEST):

- Window 30 → R² = 0.26 | DA = 0.65  
- Window 60 → R² = 0.50 | DA = 0.74  
- Window 120 → R² = 0.52 | DA = 0.74  
- Window 180 → R² = 0.57 | DA = 0.78  

Para `delta_90` (TEST):

- Window 30 → R² = 0.20 | DA = 0.63  
- Window 60 → R² = 0.36 | DA = 0.70  
- Window 120 → R² = 0.41 | DA = 0.70  
- Window 180 → R² = 0.57 | DA = 0.79  

1.2 Conclusiones sobre `delta`

- Ventanas largas mejoran sistemáticamente el desempeño.
- No hay colapso en TEST.
- VALID y TEST son consistentes.
- DA cercano a 0.78–0.79 es fuerte para un problema intradía.

Conclusión: el modelo está captando estructura real del mercado cuando el target está expresado en puntos.

---

**2. Inestabilidad del target `ret` en TEST**

2.1 Evidencia crítica

Ejemplo `ret_60` con window 180:

- VALID R² ≈ 0.48  
- TEST R² ≈ -15162  

No es simplemente un mal resultado, es una explosión del error cuadrático relativo.

2.2 Interpretación matemática

Recordando:

R² = 1 - (MSE / Var(y))

En retornos:

- La varianza del target es muy pequeña.
- Ante un outlier o cambio de régimen, el MSE crece.
- Como Var(y) es pequeña, el R² se vuelve extremadamente negativo.

Esto no implica que el modelo prediga cero, sino que el error es grande relativo a la varianza del TEST.

---

**3. Comportamiento VALID vs TEST en `ret`**

3.1 Observación clave

En VALID:

- R² ≈ 0.48 en algunos casos.
- DA ≈ 0.70.

En TEST:

- R² colapsa violentamente.
- DA se mantiene razonable (~0.70).

3.2 Interpretación

El modelo acierta dirección, pero falla magnitud cuando cambia el régimen de volatilidad.

---

**4. Comparación estructural entre targets**

- delta_60 → Alta estabilidad, buena generalización, ventana óptima ≈ 180.
- delta_90 → Alta estabilidad, buena generalización, ventana óptima ≈ 180.
- ret_60 → Baja estabilidad, generalización muy débil, comportamiento inestable.
- ret_90 → Baja estabilidad, generalización muy débil, comportamiento inestable.

Conclusión: estructuralmente, `delta` es más robusto que `ret` en este setup.

---

**5. Análisis de MAE y RMSE**

5.1 Caso `delta_60` (TEST)

- Window 30 → MAE = 50.23 | RMSE = 77.09  
- Window 60 → MAE = 38.92 | RMSE = 63.41  
- Window 90 → MAE = 40.38 | RMSE = 64.90  
- Window 120 → MAE = 39.75 | RMSE = 63.42  
- Window 180 → MAE = 38.11 | RMSE = 60.07  

Conclusiones:

- Fuerte reducción de error al pasar de 30 a 60.
- Luego estabilización.
- Window 180 es el mejor.
- RMSE no explota → comportamiento estable.

5.2 Caso `delta_90` (TEST)

- Window 30 → MAE = 64.34 | RMSE = 97.70  
- Window 60 → MAE = 56.30 | RMSE = 86.43  
- Window 90 → MAE = 56.94 | RMSE = 86.51  
- Window 120 → MAE = 56.52 | RMSE = 84.98  
- Window 180 → MAE = 49.75 | RMSE = 72.85  

Conclusiones:

- Mejora progresiva clara.
- Reducción ≈ 15 pts de MAE entre window 30 y 180.
- Coherente con la evolución del R².

Conclusión general: MAE y RMSE confirman que el modelo aprende estructura real cuando el target es `delta`.

---

**6. Análisis de MAE y RMSE en `ret`**

6.1 `ret_60` (TEST)

- Window 30 → MAE = 0.00435 | RMSE = 0.02257  
- Window 60 → MAE = 0.00617 | RMSE = 0.01858  
- Window 90 → MAE = 0.00826 | RMSE = 0.10704  
- Window 120 → MAE = 0.01908 | RMSE = 0.36743  
- Window 180 → MAE = 0.02521 | RMSE = 0.57496  

Observación:

- MAE crece gradualmente.
- RMSE explota violentamente.

Interpretación:

- Existen outliers fuertes en TEST.
- El error cuadrático amplifica eventos extremos.
- Alta sensibilidad a cambios de volatilidad.

6.2 `ret_90` (TEST)

- MAE moderado.
- RMSE nuevamente inestable y creciente en ventanas largas.

Conclusión: el problema no es direccional sino de magnitud bajo cambios de régimen.

---

**7. Conclusión técnica global**

7.1 En términos de error absoluto

- `delta` mejora consistentemente al aumentar ventana.
- `ret` se vuelve más inestable cuanto mayor es la ventana.

7.2 En términos de estabilidad estadística

- `delta` es robusto.
- `ret` es altamente sensible a varianza y régimen.

---

**8. Conclusión operativa**

Si el sistema opera en puntos reales del MNQ:

- MAE ≈ 38 pts en `delta_60` con window 180.
- Permite estimar error típico.
- Permite definir umbral mínimo operativo.
- Permite dimensionar riesgo real.

En retornos:

- El error no es homogéneo.
- Cambia violentamente entre regímenes.
- No es estable para diseño operativo directo.

---

**9. Síntesis final**

- El MLP sí aprende.
- Ventanas largas ayudan.
- `delta` es estructuralmente más estable.
- `ret` es estadísticamente frágil en este setup.
- Para robustez y generalización OOS, `delta` es claramente superior en este experimento.


## **12. Recomendación**

1. Principio correcto de selección

1.1 TRAIN → Ajusta los pesos del modelo.  
1.2 VALID → Selecciona hiperparámetros.  
1.3 TEST → Evalúa desempeño final (una sola vez).

El set TEST nunca debe influir en decisiones de arquitectura, ventana o hiperparámetros.

---

2. Entonces, ¿cómo elegir la ventana?

La elección formal debe basarse en VALID.

Si observamos VALID para `delta`:

L = 30  → R² ≈ 0.27  
L = 60  → R² ≈ 0.55  
L = 90  → R² ≈ 0.53  
L = 120 → R² ≈ 0.56  
L = 180 → R² ≈ 0.64  

La mejor ventana en VALID es claramente L = 180.

Por lo tanto:

La elección de L = 180 está justificada sin mirar TEST.

---

3. Procedimiento correcto ahora

Paso 1: Fijar L = 180 (porque VALID lo indica).  
Paso 2: Tunear hiperparámetros usando solo TRAIN + VALID.  
Paso 3: Una vez elegidos los mejores hiperparámetros, evaluar una sola vez en TEST.  

Ese resultado será tu performance OOS real.

---

4. Conclusión

Sí, la selección debe basarse en VALID.

Y en tu caso, incluso aplicando el procedimiento correcto, L = 180 sigue siendo la mejor opción estructural para `delta`.
